# Install dependencies

In [1]:
import sys
import subprocess
import importlib.util

packages = {
    "PIL": "pillow",
    "pandas": "pandas",
    "numpy": "numpy",
    "tqdm": "tqdm",
    "imgutils": "dghs-imgutils",
}

missing = [pip_name for module_name, pip_name in packages.items() if importlib.util.find_spec(module_name) is None]

if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])

print("Dependency check completed.")

Dependency check completed.


# Imports

In [2]:
from pathlib import Path
from PIL import Image, ImageOps
from datetime import datetime
from tqdm.auto import tqdm
import pandas as pd
import numpy as np
import json
import os
import sys
import platform
import warnings
import traceback
import gc

Image.MAX_IMAGE_PIXELS = 300_000_000
warnings.simplefilter("error", Image.DecompressionBombWarning)

try:
    from imgutils.metrics import ccip_batch_extract_features, ccip_extract_feature
except Exception as exc:
    raise RuntimeError(
        "Could not import CCIP feature extraction functions from dghs-imgutils. "
        "Restart the kernel after installation and run this notebook again."
    ) from exc

print("Imports completed.")

Imports completed.


# Configuration

In [3]:
PROJECT_DIR = Path("./anime_character_pipeline").expanduser().resolve()
PREVIOUS_RUN_DIR = None

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_DIR = PROJECT_DIR / "runs" / f"03_ccip_embeddings_{RUN_ID}"

REPORTS_DIR = RUN_DIR / "reports"
TABLES_DIR = RUN_DIR / "tables"
ARRAYS_DIR = RUN_DIR / "arrays"
LOGS_DIR = RUN_DIR / "logs"

CCIP_MODEL = "ccip-caformer-24-randaug-pruned"
CCIP_IMAGE_SIZE = 384
BATCH_SIZE = 16

USE_REVIEW_CROPS = True
REQUIRE_CROP_FILE_EXISTS = True
NORMALIZE_EMBEDDINGS = True

for directory in [RUN_DIR, REPORTS_DIR, TABLES_DIR, ARRAYS_DIR, LOGS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print(f"Project directory: {PROJECT_DIR}")
print(f"Run directory: {RUN_DIR}")
print(f"CCIP model: {CCIP_MODEL}")
print(f"Batch size: {BATCH_SIZE}")

Project directory: /home/ssparzival/Proyectos/GitHub/Automatic-Organization-of-Anime-Character-Images/anime_character_pipeline
Run directory: /home/ssparzival/Proyectos/GitHub/Automatic-Organization-of-Anime-Character-Images/anime_character_pipeline/runs/03_ccip_embeddings_20260523_204425
CCIP model: ccip-caformer-24-randaug-pruned
Batch size: 16


# Locate previous crop-preparation run

In [4]:
def find_latest_crop_run(project_dir):
    runs_dir = Path(project_dir) / "runs"
    candidates = sorted(
        [p for p in runs_dir.glob("02_crop_preparation_*") if p.is_dir()],
        key=lambda p: p.name,
    )
    if not candidates:
        raise FileNotFoundError(
            f"No previous crop-preparation run was found in {runs_dir}. "
            "Run notebook 02 before this notebook."
        )
    return candidates[-1]


PREVIOUS_RUN_DIR = globals().get("PREVIOUS_RUN_DIR", None)

if PREVIOUS_RUN_DIR is None:
    PREVIOUS_RUN_DIR = find_latest_crop_run(PROJECT_DIR)
else:
    PREVIOUS_RUN_DIR = Path(PREVIOUS_RUN_DIR).expanduser().resolve()

previous_tables_dir = PREVIOUS_RUN_DIR / "tables"

crop_manifest_path = previous_tables_dir / "crop_manifest_with_review_flags.csv"

if not crop_manifest_path.exists():
    raise FileNotFoundError(f"Required crop manifest was not found: {crop_manifest_path}")

crop_df = pd.read_csv(crop_manifest_path)

print(f"Previous crop-preparation run: {PREVIOUS_RUN_DIR}")
print(f"Crop manifest rows loaded: {len(crop_df)}")

Previous crop-preparation run: /home/ssparzival/Proyectos/GitHub/Automatic-Organization-of-Anime-Character-Images/anime_character_pipeline/runs/02_crop_preparation_20260523_130820
Crop manifest rows loaded: 15705


# Helper functions

In [5]:
def now_iso():
    return datetime.now().astimezone().isoformat(timespec="seconds")


def normalize_path_string(value):
    return Path(str(value)).expanduser().resolve().as_posix()


def safe_bool_series(series):
    if series.dtype == bool:
        return series
    return series.astype(str).str.lower().isin(["true", "1", "yes", "y"])


def l2_normalize_matrix(matrix, eps=1e-12):
    matrix = np.asarray(matrix, dtype=np.float32)
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    norms = np.maximum(norms, eps)
    return matrix / norms


def validate_image_file(path):
    path = Path(path)
    if not path.exists():
        return False, "FileNotFoundError", "Crop file does not exist."

    try:
        with Image.open(path) as img:
            img.verify()

        with Image.open(path) as img:
            img = ImageOps.exif_transpose(img.convert("RGB"))
            width, height = img.size

        if width <= 0 or height <= 0:
            return False, "InvalidImageSize", "Image has non-positive dimensions."

        return True, None, None

    except Exception as exc:
        return False, type(exc).__name__, str(exc)


def extract_batch_with_fallback(paths, model, size):
    paths = [str(p) for p in paths]

    try:
        features = ccip_batch_extract_features(paths, model=model, size=size)
        features = np.asarray(features, dtype=np.float32)

        if features.ndim != 2:
            raise ValueError(f"Expected 2D feature matrix, got shape {features.shape}.")

        if features.shape[0] != len(paths):
            raise ValueError(
                f"Feature row count mismatch. Expected {len(paths)}, got {features.shape[0]}."
            )

        records = []
        for idx, path in enumerate(paths):
            records.append({
                "path": path,
                "status": "ok",
                "error_type": None,
                "error_message": None,
                "feature": features[idx],
            })

        return records

    except Exception as batch_exc:
        records = []

        for path in paths:
            try:
                feature = ccip_extract_feature(path, model=model, size=size)
                feature = np.asarray(feature, dtype=np.float32)

                if feature.ndim != 1:
                    raise ValueError(f"Expected 1D feature vector, got shape {feature.shape}.")

                records.append({
                    "path": path,
                    "status": "ok",
                    "error_type": None,
                    "error_message": None,
                    "feature": feature,
                })

            except Exception as single_exc:
                records.append({
                    "path": path,
                    "status": "error",
                    "error_type": type(single_exc).__name__,
                    "error_message": str(single_exc),
                    "feature": None,
                })

        return records


def batch_iterable(values, batch_size):
    values = list(values)
    for start in range(0, len(values), batch_size):
        yield values[start:start + batch_size]


print("Helper functions loaded.")

Helper functions loaded.


# Select crops for embedding


In [6]:
required_columns = {"crop_path", "status"}

missing_columns = sorted(required_columns.difference(set(crop_df.columns)))

if missing_columns:
    raise ValueError(f"The crop manifest is missing required columns: {missing_columns}")

embedding_input_df = crop_df.copy()
embedding_input_df["crop_path"] = embedding_input_df["crop_path"].map(normalize_path_string)

if "needs_review" in embedding_input_df.columns:
    embedding_input_df["needs_review"] = safe_bool_series(embedding_input_df["needs_review"])
else:
    embedding_input_df["needs_review"] = False

embedding_input_df = embedding_input_df[embedding_input_df["status"].eq("ok")].copy()

if not USE_REVIEW_CROPS:
    embedding_input_df = embedding_input_df[~embedding_input_df["needs_review"]].copy()

if REQUIRE_CROP_FILE_EXISTS:
    embedding_input_df["crop_file_exists"] = embedding_input_df["crop_path"].map(lambda p: Path(p).exists())
    embedding_input_df = embedding_input_df[embedding_input_df["crop_file_exists"]].copy()
else:
    embedding_input_df["crop_file_exists"] = embedding_input_df["crop_path"].map(lambda p: Path(p).exists())

embedding_input_df = embedding_input_df.reset_index(drop=True)
embedding_input_df.insert(0, "embedding_index", np.arange(len(embedding_input_df), dtype=int))

embedding_input_path = TABLES_DIR / "embedding_input_manifest.csv"
embedding_input_df.to_csv(embedding_input_path, index=False)

print(f"Crop rows loaded: {len(crop_df)}")
print(f"Crops selected for embedding: {len(embedding_input_df)}")

if embedding_input_df.empty:
    raise RuntimeError("No crops were selected for embedding extraction.")

Crop rows loaded: 15705
Crops selected for embedding: 15705


# Validate selected crop files before CCIP


In [7]:
validation_rows = []

for _, row in tqdm(embedding_input_df.iterrows(), total=len(embedding_input_df), desc="Validating crop files"):
    is_valid, error_type, error_message = validate_image_file(row["crop_path"])
    validation_rows.append({
        "embedding_index": int(row["embedding_index"]),
        "crop_path": row["crop_path"],
        "valid_for_embedding": bool(is_valid),
        "validation_error_type": error_type,
        "validation_error_message": error_message,
    })

crop_validation_df = pd.DataFrame(validation_rows)

embedding_ready_df = embedding_input_df.merge(
    crop_validation_df,
    on=["embedding_index", "crop_path"],
    how="left",
)

embedding_ready_df = embedding_ready_df[embedding_ready_df["valid_for_embedding"].astype(bool)].copy()
embedding_ready_df = embedding_ready_df.reset_index(drop=True)
embedding_ready_df["embedding_index"] = np.arange(len(embedding_ready_df), dtype=int)

invalid_crop_validation_df = crop_validation_df[~crop_validation_df["valid_for_embedding"].astype(bool)].copy()

crop_validation_path = TABLES_DIR / "crop_embedding_validation.csv"
embedding_ready_path = TABLES_DIR / "embedding_ready_manifest.csv"
invalid_crop_validation_path = TABLES_DIR / "invalid_crops_for_embedding.csv"

crop_validation_df.to_csv(crop_validation_path, index=False)
embedding_ready_df.to_csv(embedding_ready_path, index=False)
invalid_crop_validation_df.to_csv(invalid_crop_validation_path, index=False)

print(f"Valid crops for embedding: {len(embedding_ready_df)}")
print(f"Invalid crops for embedding: {len(invalid_crop_validation_df)}")

if embedding_ready_df.empty:
    raise RuntimeError("No valid crop files are available for CCIP embedding extraction.")

Validating crop files:   0%|          | 0/15705 [00:00<?, ?it/s]

Valid crops for embedding: 15705
Invalid crops for embedding: 0


# Warm up CCIP model on the first crop

In [8]:
warmup_path = embedding_ready_df.iloc[0]["crop_path"]

try:
    warmup_feature = ccip_extract_feature(warmup_path, model=CCIP_MODEL, size=CCIP_IMAGE_SIZE)
    warmup_feature = np.asarray(warmup_feature, dtype=np.float32)
    print(f"Warmup feature shape: {warmup_feature.shape}")
except Exception as exc:
    raise RuntimeError(
        "CCIP warmup failed. If this is the first run, check that the machine has internet access "
        "to download the model, then retry."
    ) from exc

if warmup_feature.ndim != 1:
    raise ValueError(f"Expected a one-dimensional CCIP feature vector, got shape {warmup_feature.shape}.")

print("CCIP warmup completed.")

ccip-caformer-24-randaug-pruned/model_fe(…):   0%|          | 0.00/150M [00:00<?, ?B/s]

Warmup feature shape: (768,)
CCIP warmup completed.


# Extract CCIP embeddings

In [9]:
all_paths = embedding_ready_df["crop_path"].tolist()
feature_records = []

for batch_paths in tqdm(list(batch_iterable(all_paths, BATCH_SIZE)), desc="Extracting CCIP embeddings"):
    batch_records = extract_batch_with_fallback(
        paths=batch_paths,
        model=CCIP_MODEL,
        size=CCIP_IMAGE_SIZE,
    )
    feature_records.extend(batch_records)
    gc.collect()

feature_status_df = pd.DataFrame([
    {
        "crop_path": record["path"],
        "embedding_status": record["status"],
        "embedding_error_type": record["error_type"],
        "embedding_error_message": record["error_message"],
    }
    for record in feature_records
])

ok_records = [record for record in feature_records if record["status"] == "ok"]
error_records = [record for record in feature_records if record["status"] != "ok"]

print(f"Feature records: {len(feature_records)}")
print(f"Successful embeddings: {len(ok_records)}")
print(f"Embedding errors: {len(error_records)}")

if not ok_records:
    raise RuntimeError("All CCIP embedding extractions failed.")

Extracting CCIP embeddings:   0%|          | 0/982 [00:00<?, ?it/s]

Feature records: 15705
Successful embeddings: 15705
Embedding errors: 0


# Assemble raw and normalized embedding matrices


In [13]:
ok_paths = [record["path"] for record in ok_records]
raw_embeddings = np.vstack([record["feature"] for record in ok_records]).astype(np.float32)

if raw_embeddings.ndim != 2:
    raise ValueError(f"Expected a 2D embedding matrix, got shape {raw_embeddings.shape}.")

if raw_embeddings.shape[0] != len(ok_paths):
    raise ValueError("Embedding matrix row count does not match successful path count.")

embedding_dim = int(raw_embeddings.shape[1])

if NORMALIZE_EMBEDDINGS:
    normalized_embeddings = l2_normalize_matrix(raw_embeddings)
else:
    normalized_embeddings = raw_embeddings.copy()

norms_before = np.linalg.norm(raw_embeddings, axis=1)
norms_after = np.linalg.norm(normalized_embeddings, axis=1)

print(f"Raw embedding matrix shape: {raw_embeddings.shape}")
print(f"Normalized embedding matrix shape: {normalized_embeddings.shape}")
print(f"Embedding dimension: {embedding_dim}")
print(f"Raw norm range: {float(norms_before.min()):.6f} to {float(norms_before.max()):.6f}")
print(f"Final norm range: {float(norms_after.min()):.6f} to {float(norms_after.max()):.6f}")

Raw embedding matrix shape: (15705, 768)
Normalized embedding matrix shape: (15705, 768)
Embedding dimension: 768
Raw norm range: 40.324081 to 506.429260
Final norm range: 1.000000 to 1.000000


# Create final embedding manifest

In [15]:
successful_embedding_df = pd.DataFrame({
    "embedding_row": np.arange(len(ok_paths), dtype=int),
    "crop_path": ok_paths,
})

if successful_embedding_df["crop_path"].duplicated().any():
    duplicated_paths = successful_embedding_df.loc[
        successful_embedding_df["crop_path"].duplicated(keep=False),
        "crop_path"
    ].drop_duplicates().head(10).tolist()

    raise ValueError(
        "Duplicated crop_path values were found in successful_embedding_df. "
        f"Examples: {duplicated_paths}"
    )

feature_status_df_clean = feature_status_df.copy()

if feature_status_df_clean["crop_path"].duplicated().any():
    feature_status_df_clean = feature_status_df_clean.drop_duplicates(
        subset=["crop_path"],
        keep="first",
    ).reset_index(drop=True)

embedding_manifest_df = embedding_ready_df.merge(
    feature_status_df_clean,
    on="crop_path",
    how="left",
    validate="one_to_one",
)

embedding_manifest_df["embedding_status"] = embedding_manifest_df["embedding_status"].fillna("not_processed")

successful_embedding_manifest_df = embedding_manifest_df[
    embedding_manifest_df["embedding_status"].eq("ok")
].copy()

successful_embedding_manifest_df = successful_embedding_manifest_df.merge(
    successful_embedding_df,
    on="crop_path",
    how="left",
    validate="one_to_one",
)

successful_embedding_manifest_df = successful_embedding_manifest_df.sort_values(
    "embedding_row"
).reset_index(drop=True)

if successful_embedding_manifest_df["embedding_row"].isna().any():
    missing_rows = successful_embedding_manifest_df[
        successful_embedding_manifest_df["embedding_row"].isna()
    ][["crop_path", "relative_path"]].head(10)

    raise ValueError(
        "Some successful embeddings do not have an embedding_row assigned. "
        f"Examples:\n{missing_rows}"
    )

successful_embedding_manifest_df["embedding_row"] = successful_embedding_manifest_df["embedding_row"].astype(int)

if len(successful_embedding_manifest_df) != raw_embeddings.shape[0]:
    raise ValueError(
        "Successful embedding manifest row count does not match embedding matrix row count. "
        f"Manifest rows: {len(successful_embedding_manifest_df)}. "
        f"Embedding rows: {raw_embeddings.shape[0]}."
    )

if raw_embeddings.shape != normalized_embeddings.shape:
    raise ValueError(
        "Raw and normalized embedding matrices have different shapes. "
        f"Raw: {raw_embeddings.shape}. Normalized: {normalized_embeddings.shape}."
    )

expected_rows = list(range(raw_embeddings.shape[0]))
actual_rows = successful_embedding_manifest_df["embedding_row"].tolist()

if actual_rows != expected_rows:
    raise ValueError("embedding_row is not consecutive after sorting.")

print(f"Full embedding manifest rows: {len(embedding_manifest_df)}")
print(f"Successful embedding manifest rows: {len(successful_embedding_manifest_df)}")
print(f"Embedding matrix rows: {raw_embeddings.shape[0]}")
print(f"Embedding dimension: {raw_embeddings.shape[1]}")

Full embedding manifest rows: 15705
Successful embedding manifest rows: 15705
Embedding matrix rows: 15705
Embedding dimension: 768


# Save arrays and manifests

In [16]:
raw_embeddings_path = ARRAYS_DIR / "ccip_embeddings_raw.npy"
normalized_embeddings_path = ARRAYS_DIR / "ccip_embeddings_l2.npy"
compressed_embeddings_path = ARRAYS_DIR / "ccip_embeddings_bundle.npz"

embedding_manifest_path = TABLES_DIR / "embedding_manifest.csv"
successful_embedding_manifest_path = TABLES_DIR / "successful_embedding_manifest.csv"
feature_status_path = TABLES_DIR / "embedding_status.csv"

np.save(raw_embeddings_path, raw_embeddings)
np.save(normalized_embeddings_path, normalized_embeddings)

np.savez_compressed(
    compressed_embeddings_path,
    raw_embeddings=raw_embeddings,
    normalized_embeddings=normalized_embeddings,
)

embedding_manifest_df.to_csv(embedding_manifest_path, index=False)
successful_embedding_manifest_df.to_csv(successful_embedding_manifest_path, index=False)
feature_status_df.to_csv(feature_status_path, index=False)

print(f"Saved raw embeddings: {raw_embeddings_path}")
print(f"Saved normalized embeddings: {normalized_embeddings_path}")
print(f"Saved compressed bundle: {compressed_embeddings_path}")
print(f"Saved embedding manifest: {embedding_manifest_path}")
print(f"Saved successful embedding manifest: {successful_embedding_manifest_path}")

Saved raw embeddings: /home/ssparzival/Proyectos/GitHub/Automatic-Organization-of-Anime-Character-Images/anime_character_pipeline/runs/03_ccip_embeddings_20260523_204425/arrays/ccip_embeddings_raw.npy
Saved normalized embeddings: /home/ssparzival/Proyectos/GitHub/Automatic-Organization-of-Anime-Character-Images/anime_character_pipeline/runs/03_ccip_embeddings_20260523_204425/arrays/ccip_embeddings_l2.npy
Saved compressed bundle: /home/ssparzival/Proyectos/GitHub/Automatic-Organization-of-Anime-Character-Images/anime_character_pipeline/runs/03_ccip_embeddings_20260523_204425/arrays/ccip_embeddings_bundle.npz
Saved embedding manifest: /home/ssparzival/Proyectos/GitHub/Automatic-Organization-of-Anime-Character-Images/anime_character_pipeline/runs/03_ccip_embeddings_20260523_204425/tables/embedding_manifest.csv
Saved successful embedding manifest: /home/ssparzival/Proyectos/GitHub/Automatic-Organization-of-Anime-Character-Images/anime_character_pipeline/runs/03_ccip_embeddings_20260523_204